# Lesson 22 · Derivatives — teaching a computer to "feel change": tangent slope and numerical differentiation by central difference

**Learning goals**
1. Understand derivative = instantaneous rate of change / tangent slope
2. Do numerical differentiation with the central difference, and know that it is usually more accurate than the forward difference (error order roughly $O(h^2)$ vs $O(h)$)
3. Be able to check the code against hand calculations on small functions (such as $f(x)=x^2$)

**Why it matters for Aurora**: numerical derivatives are the ruler for checking whether backpropagation "got it right" — analytical gradient vs numerical gradient, compared answer for answer.

← **Previous lesson** · [L21 · Matrices as filters](../2_linear_algebra/L21_aurora_matrices.ipynb)

> Last lesson covered **matrices as filters**: the DFT matrix and the mel matrix — audio processing is matrix multiplication.
> This lesson explores **derivatives**.

## This lesson's story: how do you teach a computer to "feel change"?

Humans have an intuition for "slope" — stand on a hillside and your feet feel the tilt.
But a computer only has numbers. How does it know how "steep" a function is at a point?

The method: take two points very close to x, draw a secant line through them, and use the secant's slope to approximate the tangent slope:

```
f'(x) ≈ (f(x+h) - f(x-h)) / (2h)     ← central difference formula
```

The closer the two points (the smaller h), the closer the secant gets to the tangent. But h can't be infinitely small — floating-point numbers have a precision floor (about 1e-15), and too small an h causes "catastrophic cancellation", which makes the error bigger instead.

The best h ≈ 1e-5: not too big, not too small, balancing truncation error and rounding error.

Why does this matter for Aurora?
Gradient descent (L25) needs a derivative at every parameter. The numerical derivative `numeric_derivative` is a "universal" gradient checker — it verifies whether your analytic formula has a mistake, and it's a basic tool for debugging ML systems.

## Learning goals

1. **State the limit definition**: write f'(x) = lim_{h→0} (f(x+h)-f(x))/h, and explain why a computer has to approximate it with a finite h.
2. **Derive the O(h²) error**: explain why the central difference is one order more accurate than the forward difference — the error drops from O(h) to O(h²), about 5 orders of magnitude smaller at h=1e-5 (Taylor expansion cancels the first-order term).
3. **Implement `numeric_derivative`**: complete the TODO stub, using the central difference to approximate the derivative at any point.
4. **Find the best h by experiment**: in an h-sweep experiment, watch the error fall and then rise in a U-shaped curve, and confirm the best step size is about 1e-5 to 1e-7.

## 1. Central difference: approximating the slope with two points

**Why use the central difference rather than the forward difference?**

Forward difference: `f'(x) ≈ (f(x+h) - f(x)) / h`, error order O(h)
Central difference: `f'(x) ≈ (f(x+h) - f(x-h)) / (2h)`, error order O(h²)

At h=1e-5, the forward error ≈ 1e-5 and the central-difference error ≈ 1e-10 — **5 orders of magnitude** smaller.

Check by hand (f(x)=x², x=1, h=0.1):
```
CD = (f(1.1) - f(0.9)) / (0.2)
   = (1.21 - 0.81) / 0.2
   = 0.40 / 0.2
   = 2.0      ← exactly the true value f'(1) = 2×1 = 2!
```

**Why can't h be infinitely small?**
Floating point has a machine precision ε ≈ 2.2×10⁻¹⁶. When h < √ε ≈ 1e-8, the numerator `f(x+h)-f(x-h)` is tiny compared with `f(x)`, and significant digits are lost (catastrophic cancellation).

### Taylor expansion: getting to the bottom of "cancelling the first-order term"

The hand calculation above only checked one special example (`f(x)=x²`), and it happened to come out "exactly right" — that's because `x²` has no higher-order "bending" of its own. For a general function we need another tool to explain "why the central difference is exactly one order more accurate than the forward difference": the **Taylor expansion**.

**What does a Taylor expansion say?** Imagine you know someone's position, speed and acceleration right now… then you can estimate roughly where they'll be after a small step `h`: first go by "position", correct once with "speed × time", correct again with "half of acceleration × time²", and so on, each correction smaller than the last. That's the intuition behind the Taylor expansion — use the rates of change of every order at the current point to predict roughly what the function's value will be "one small step over":

$$f(x+h) = f(x) + f'(x)h + \frac{f''(x)}{2}h^2 + \frac{f'''(x)}{6}h^3 + \frac{f''''(x)}{24}h^4 + \cdots$$

The smaller h is, the faster the later terms fall away (h⁴ is much smaller than h²), so the first few terms are enough.

**Now replace h with -h and compare:**

$$f(x-h) = f(x) - f'(x)h + \frac{f''(x)}{2}h^2 - \frac{f'''(x)}{6}h^3 + \frac{f''''(x)}{24}h^4 - \cdots$$

Notice the pattern: **even powers** of h (h², h⁴, …) keep their sign (because `(-h)² = h²`), while **odd powers** of h (h¹, h³, …) flip sign (because `(-h)³ = -h³`).

**Subtract the two** (exactly what the central difference's numerator does):

$$f(x+h) - f(x-h) = 2f'(x)h + \frac{2f'''(x)}{6}h^3 + O(h^5) = 2f'(x)h + \frac{f'''(x)}{3}h^3 + O(h^5)$$

The even-power terms (the h² term with `f''`) have the same sign in both expansions, so they **cancel completely** on subtraction — that's what "cancelling the first-order term" really means: not cancelling f' itself, but cancelling the `f''h²` term that comes right after f' and would otherwise pollute the result. The lowest-order error term left is the next odd-power term: `f'''(x)h³/3`.

**Divide both sides by 2h:**

$$\frac{f(x+h)-f(x-h)}{2h} = f'(x) + \frac{f'''(x)}{6}h^2 + O(h^4)$$

The central difference's result = the true derivative + an error term proportional to h². That's where **O(h²)** comes from directly.

**Compare with the forward difference**: `f(x+h)-f(x)` uses the expansion on one side only, with no other half to cancel the `f''h²` term:

$$\frac{f(x+h)-f(x)}{h} = f'(x) + \frac{f''(x)}{2}h + O(h^2)$$

The error term is proportional to h (**O(h)**), a whole order of h worse than the central difference — which answers "why does symmetry cancel one more factor of h": thanks to its left–right symmetry, the central difference gets an entire term cancelled for free.

Back to the original `f(x)=x²` example to check: `f'(x)=2x`, `f''(x)=2`, `f'''(x)=0` (a quadratic's third derivative is always 0). Plug into the error formula above: the central difference's error term `f'''(x)h²/6 = 0` — **the error is exactly zero**. That's no coincidence; it's because the third derivative of `x²` is 0 to begin with, which is why the hand calculation came out "exactly right". Below, the code checks on a general function (`sin`) that the forward and central differences' errors really do shrink at rates an order apart as h gets smaller:

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np


def forward_diff(f, x, h):
    return (f(x + h) - f(x)) / h


def central_diff(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)


x0 = 1.0
true_val = np.cos(x0)  # f=sin(x), f'(x)=cos(x)

print(f"{'h':>10} {'forward err':>14} {'central err':>14}")
h = 0.1
for _ in range(5):
    fd_err = abs(forward_diff(np.sin, x0, h) - true_val)
    cd_err = abs(central_diff(np.sin, x0, h) - true_val)
    print(f"{h:10.0e} {fd_err:14.3e} {cd_err:14.3e}")
    h /= 2

print("\nEach time h halves: the forward error roughly halves (linear, O(h)); the central error drops to about 1/4 (quadratic, O(h²)) —")
print("exactly the orders the Taylor expansion above predicts; no coincidence.")

### The limit definition

The rigorous definition of the derivative comes from a limit:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

It means: the closer h gets to 0, the closer the slope of the secant line gets to the slope of the tangent line.

**But a computer can't take h→0**: floating-point precision is limited (about 1e-16), and when h is too small, the numerator `f(x+h)-f(x-h)` subtracts two nearly equal numbers and significant digits are lost rapidly (catastrophic cancellation).

So in practice we approximate with the **central difference** at a finite h (usually h≈1e-5):

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}, \quad h \approx 10^{-5}$$

This squeezes the error from O(h) (forward difference) down to O(h²) (central difference), about 1e-10 at h=1e-5.

### What does "lim" actually mean? — a speedometer analogy

The `lim_{h→0}` in the formula looks like some mysterious symbol, but what it says is very down to earth.

Imagine you want to know a bicycle's speed at **one instant**. You can't measure "a single moment" (the time elapsed would be 0 and so would the distance covered, and `0/0` can't be computed). But you can measure how far it travels in 1 second, in 0.1 seconds, in 0.01 seconds… and compute the **average speed** over each window. As the time window shrinks, the measured average speed settles down more and more, finally "closing in" on a fixed number — that number is the "true speed" at that instant.

`lim_{h→0} g(h)` is read "the limit of g(h) as h tends to 0". It doesn't ask "what is g(h) when h equals 0" (often the undefined `0/0`), but "as h gets closer and closer to 0 (without ever actually equaling 0), which number does g(h) steadily approach?".

Applied to derivatives: `g(h) = (f(x+h)-f(x))/h` is the "secant slope", and the number this ratio steadily approaches as h shrinks is defined to be `f'(x)` — the curve's "instantaneous slope" at that point, exactly like the speedometer.

**So what does a computer do?** The mathematical `lim` lets h shrink without end, but a computer's floating-point numbers have a precision floor (about 2.2×10⁻¹⁶), so h can't really go to 0 — shrinking it too far actually makes the result worse through "catastrophic cancellation" (demonstrated concretely below). So in engineering we don't chase a "mathematically ideal" infinitesimal h; we look for a **finite h with the smallest total error**: as long as the total error (truncation error + rounding error) is below the precision you actually need — for example, Aurora's gradient check uses 1e-5 as its pass threshold — this h is "small enough" to stand in for the limit. The "parameter experiment" section later works out exactly how this best h is derived.

## Into the lab: watching a function through numerical change

The experiment below computes the numerical derivative of f(x)=x² at x=3.0; watch how the computed value approaches the true slope (true value = 2x = 6) as `h` shrinks.

In [ ]:
import numpy as np
f = lambda x: x**2
h = 1e-5
x = 3.0
print('slope of f(x)=x^2 at x=3 ≈', (f(x+h)-f(x-h))/(2*h), ' (true value 2x=6)')

In [ ]:
# Tangent visualization: the function curve + the tangent line
import numpy as np
import matplotlib.pyplot as plt

x_arr = np.linspace(0, 5, 300)
f_arr = x_arr**2

x0 = 3.0
slope = 2 * x0  # f'(x0) = 2x0 = 6 (analytic value)
tangent = f(x0) + slope * (x_arr - x0)  # tangent line equation

plt.figure(figsize=(6, 4))
plt.plot(x_arr, f_arr, label=r"$f(x)=x^2$", linewidth=2)
plt.plot(x_arr, tangent, "--", label=f"tangent @ x={x0} (slope={slope})", linewidth=1.5)
plt.scatter([x0], [x0**2], color="red", zorder=5, label=f"x={x0}")
plt.xlim(0, 5); plt.ylim(-5, 25)
plt.xlabel("x"); plt.ylabel("f(x)")
plt.title("Tangent slope = value of the derivative")
plt.legend(); plt.tight_layout(); plt.show()
print(f"At x={x0}: analytic slope={slope}; the numerical slope (once implemented) is compared in the check cell")

## Hands-on: slopes at several points at once

Compute the numerical slopes of f(x)=x² at the five points xs=[-2,-1,0,1,2] in one batch (a vectorized central difference) and compare them point by point with the true values 2x: they match to the 3 decimal places printed (for x² the central difference is exact up to floating-point rounding; here h=1e-3).

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('approximate slope =', np.round(slopes, 3))
print('theoretical slope 2x =', 2 * xs)


## Code experiment: step through different positions and watch how the slope changes

Take 7 points of f(x)=x²+2x on [-3, 3] and print the numerical derivative at each one; compare it with the analytic derivative 2x+2 to confirm that the central difference is accurate across the whole interval.

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ Implement `numeric_derivative(f, x, h=1e-5)`

**Line of reasoning**:
1. Function signature: takes `f` (any callable), `x` (float, the point to differentiate at) and `h` (perturbation step, default 1e-5).
2. Compute `f(x+h)` and `f(x-h)` — perturbing by h to the right and to the left, two function evaluations in all.
3. Subtract the two and divide by `2*h` to get the central-difference approximation; return a single float.

**Reference input and output**: f=sin, x=0, h=1e-5 → `(sin(1e-5) - sin(-1e-5))/(2e-5)` ≈ `(1e-5 - (-1e-5))/(2e-5)` = 1.0 = cos(0) ✓

> 💡 Need a hint? After the exercise, you can look at the reference implementation in the `solutions/` folder.


### Before you write code, fill in the variable table

Before writing `numeric_derivative`, be clear about three things:
- Input: `f` (a callable), `x` (the point to differentiate at), `h` (perturbation step, default 1e-5)
- Key step: compute `f(x+h)` and `f(x-h)`, subtract, and divide by `2h`
- Return: a single float, the approximate slope of f at x

In [ ]:
def numeric_derivative(f, x, h=1e-5):
    # ✏️ TODO: approximate the derivative with a central difference
    raise NotImplementedError("TODO: implement the central-difference approximation — use (f(x+h) - f(x-h)) / (2*h)")

In [ ]:
try:
    assert abs(numeric_derivative(lambda x: x**2, 3.0) - 6.0) < 1e-8, "the derivative of x² at x=3 should be ≈6.0, with error <1e-8"
    assert abs(numeric_derivative(np.sin, 0.0) - 1.0) < 1e-8, "the derivative of sin at x=0 should be ≈1.0, with error <1e-8"
    print("✅ Passed: you can differentiate numerically.")
except (NotImplementedError, TypeError):
    print("⚠️ numeric_derivative isn't implemented yet; finish the TODO and rerun.")
except AssertionError as e:
    print(f"❌ Assertion failed: {e}")

**A quick question: why do the asserts use `< 1e-8` instead of requiring "exactly equal"?**

If you run the asserts above, you'll find that `numeric_derivative(np.sin, 0.0)` may come out as something like `0.999999999983333`, not a clean `1.0`. That's not a bug; it's the inevitable result of two errors stacking up:

1. The central difference itself has an `O(h²)` truncation error — around 1e-10 at h=1e-5 (derived with the Taylor expansion in the previous section);
2. Computing `sin` and doing the subtraction add an even smaller floating-point rounding error.

Mixed together, the final result differs a little from the analytic value `1.0`, usually by something in the `1e-14 ~ 1e-10` range — **exact equality is neither possible nor needed**.

The `1e-8` threshold is chosen several orders of magnitude looser than the "normal error" (about 1e-10), a deliberate safety margin:

- If `numeric_derivative` is implemented correctly, the error is around 1e-10 and easily passes `<1e-8`;
- If the implementation is **wrong** (say a plus and minus swapped, or forgetting to divide by `2h`), the computed value is off from the truth by something of order `O(1)` (say 0.5 or more), far beyond 1e-8, and the assert fails.

So the job of `1e-8` is "catch real bugs", not "demand numerical perfection" — a very common practice in numerical computing: make the tolerance at least an order of magnitude looser than the expected error, so normal fluctuations across different functions and different h aren't mistaken for errors.

## 3. Learn a few common derivatives (memorizing them saves time)

| f(x) | f'(x) |
|---|---|
| xⁿ | n·xⁿ⁻¹ |
| eˣ | eˣ |
| sin x | cos x |
| ln x | 1/x |

### Where do these formulas come from? — deriving two examples by hand from the "secant"

This table looks like a set of "magic formulas" to cram, but every line can be derived by hand from the "secant slope" idea we started with — still using `[f(x+h)-f(x)]/h`, only this time letting h really go to 0 (not the computer's approximation).

**The case of xⁿ: starting from the binomial expansion**

Take `n=2`, `f(x)=x²`:

```
(x+h)² = x² + 2xh + h²
[f(x+h)-f(x)]/h = (2xh + h²)/h = 2x + h
```

As h→0, `2x + h → 2x` — that's `f'(x)=2x`.

Now `n=3`, `f(x)=x³`:

```
(x+h)³ = x³ + 3x²h + 3xh² + h³
[f(x+h)-f(x)]/h = 3x² + 3xh + h²
```

As h→0 → `3x²`, i.e. `f'(x)=3x²`.

The pattern is clear: the second term of the binomial expansion of `(x+h)ⁿ` is always `n·xⁿ⁻¹·h` (fixed by the binomial coefficients), and every other term carries at least `h²` or a higher power, so they all vanish as h→0, leaving only `n·xⁿ⁻¹`. That's where `xⁿ → n·xⁿ⁻¹` comes from — and it holds for any positive integer n.

**The case of sin x: starting from the angle-addition formula**

Use the trig identity to split `sin(x+h)`:

```
sin(x+h) = sin(x)cos(h) + cos(x)sin(h)
```

When h is small (in radians), what do `cos(h)` and `sin(h)` approach? `cos(h)` barely changes near 0 (`cos(0)=1`, and cos is an even function whose first-order rate of change is 0, so `cos(h)≈1`); `sin(h)≈h` — a basic geometric fact about radians: for small h, a short arc and its corresponding straight line are almost the same length, so `sin(h)/h → 1`.

Substituting:

```
sin(x+h) ≈ sin(x)·1 + cos(x)·h = sin(x) + h·cos(x)
[sin(x+h)-sin(x)]/h ≈ cos(x)
```

As h→0 the error terms vanish and we get `f'(x)=cos(x)` — that's where `sin x → cos x` comes from. `eˣ → eˣ` and `ln x → 1/x` can be derived with similar limit tricks; for now, remember the results and check them with numerical derivatives:

In [ ]:
# Check: do the analytic derivative formulas for xⁿ and sin x agree with the numerical derivatives?
try:
    for n in [2, 3, 4]:
        f_n = lambda x, n=n: x**n
        df_n_analytical = lambda x, n=n: n * x ** (n - 1)
        x0 = 2.0
        nd = numeric_derivative(f_n, x0)
        print(f"f(x)=x^{n}  analytic f'({x0})={df_n_analytical(x0):.4f}  numerical={nd:.4f}")

    x0 = 0.7
    nd_sin = numeric_derivative(np.sin, x0)
    print(f"\nf(x)=sin(x)  analytic f'({x0})=cos({x0})={np.cos(x0):.6f}  numerical={nd_sin:.6f}")
except (NotImplementedError, TypeError):
    print("⬜ Finish the numeric_derivative exercise above first, then run this cell to check the derivative table")

**🔗 Aurora connection**: in a later milestone, `numeric_derivative` will be integrated into `src/aurora/audio/grad_check.py` (not created yet), where it will compare the analytic gradients of modules such as MFCC and the mel filterbank with numerical gradients entry by entry, like an automatic health check for your derivative code; a difference above 1e-5 usually means the implementation needs a look. For now, you can use the function you implemented directly in this notebook to check things. The next lesson (`L23_gradients.ipynb`) extends the single-point slope to many variables: repeat `numeric_derivative` once along each parameter dimension and stack the results into a gradient vector.

In [ ]:
# Parameter experiment: the best range for h (matches the "Parameter experiment" description below)
hs = [1e-1, 1e-2, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11, 1e-13, 1e-15]
try:
    errs = [abs(numeric_derivative(np.sin, 1.0, h) - np.cos(1.0)) for h in hs]  # exact value cos(1)
except (NotImplementedError, TypeError):
    print("⬜ Finish the numeric_derivative exercise above first, then run this cell")
else:
    print(f"{'h':>10}  {'|error|':>12}")
    print('-' * 26)
    for h, err in zip(hs, errs):
        print(f"{h:10.0e}  {err:12.2e}")
    print("\nThe best h is about 1e-5 to 1e-7, with error about 1e-11; shrink h further and floating-point rounding error rises instead.")


In [ ]:
# A concrete example: when h is too small, how many significant digits does the subtraction actually lose?
x0 = 1.0
print(f"{'h':>8} | {'f(x+h)':>21} | {'f(x-h)':>21} | {'difference':>12}")
print('-' * 70)
for h_val in [1e-2, 1e-6, 1e-10, 1e-14, 1e-16, 1e-17]:
    a = float(np.sin(x0 + h_val))
    b = float(np.sin(x0 - h_val))
    print(f"{h_val:8.0e} | {a:.17f} | {b:.17f} | {a - b:12.6e}")

print("""
Line up f(x+h) and f(x-h) by decimal place: at h=1e-14 the first 14 digits of the two numbers are identical,
and they only start to differ from the 14th–15th digit — in the subtraction, all the matching high digits cancel, leaving only the last
one or two "significant digits" to compute the derivative with; a dozen-plus of the 17 digits of precision are wasted.
At h=1e-16 almost every digit lines up, leaving just 1 significant digit; at h=1e-17 the two floating-point numbers
have been rounded to exactly the same bits, so a-b is simply 0.0 — the numerator becomes 0, and the computed
"derivative" becomes 0 too, completely wrong! This is "catastrophic cancellation": the formula isn't wrong; floating-point numbers
just can't remember such a fine difference.
""")

### How exactly do we get the best h≈1e-5?

All three students asked about the same point: "truncation error and rounding error trade off against each other" makes sense, but where does the specific number 1e-5 come from? The answer: write both errors as functions of h, add them up, and find the minimum.

**Truncation error**: the Taylor expansion earlier showed that the central difference's error term is `f'''(x)h²/6`, written as:

$$E_{\text{trunc}}(h) \approx \frac{|f'''(x)|}{6}h^2$$

The smaller h is, the smaller this term.

**Rounding error**: every time `f(x)` is computed, floating-point storage carries "noise" of about `ε·|f(x)|` (`ε≈2.2×10⁻¹⁶` is the machine precision, the smallest relative gap between two neighboring floating-point numbers). `f(x+h)` and `f(x-h)` each carry their own share of noise, the subtraction doesn't cancel it, and dividing by `2h` amplifies it further:

$$E_{\text{round}}(h) \approx \frac{\varepsilon\,|f(x)|}{h}$$

The smaller h is, the smaller the denominator and the *larger* this term — this is what "catastrophic cancellation" looks like in the error formula (the cell above demonstrated this cancellation with concrete numbers).

**Add the two terms and you have the total error:**

$$E(h) \approx \frac{|f'''(x)|}{6}h^2 + \frac{\varepsilon\,|f(x)|}{h}$$

It's a tug of war — "as h shrinks, the first term shrinks and the second grows" — so there has to be a minimum in between. Finding the minimum means differentiating with respect to h and setting the derivative to 0:

$$\frac{dE}{dh} \approx \frac{|f'''(x)|}{3}h - \frac{\varepsilon\,|f(x)|}{h^2} = 0 \quad\Longrightarrow\quad h_{\text{opt}} \approx \left(\frac{3\varepsilon\,|f(x)|}{|f'''(x)|}\right)^{1/3}$$

For functions like `sin` and `cos`, `|f(x)|` and `|f'''(x)|` are both of order `[-1,1]`, so their ratio is about 1; plugging in `ε≈2.2×10⁻¹⁶`:

$$h_{\text{opt}} \approx (3\varepsilon)^{1/3} \approx (6.6\times10^{-16})^{1/3} \approx 8.7\times10^{-6} \approx 1\times10^{-5}$$

That's exactly the "best h≈1e-5" seen in the h-sweep experiment — not a number pulled out of thin air, but the crossing point of the two curves "truncation error ∝ h²" and "rounding error ∝ 1/h". Plugging `h_opt` back in also gives the order of the minimum total error, `≈ε^(2/3)≈3.7×10⁻¹¹`, which matches the error of about 1e-11 in the experiment too.

**A small reminder**: earlier we said "catastrophic cancellation starts when h<√ε≈1e-8"; that's a rougher rule of thumb (marking where the subtraction "starts noticeably" losing precision), not the point where the total error is truly smallest. Because the central difference's truncation error is of order `h²` (converging much faster than the forward difference's order `h`), the true best h is of the order of the **cube root** of `ε`, several orders of magnitude larger than `√ε` — which is also why the best h measured in the experiment (1e-5 to 1e-7) is noticeably larger than `√ε` (about 1e-8).

In [ ]:
eps = np.finfo(float).eps
h_opt_estimate = (3 * eps) ** (1 / 3)
print(f"machine precision ε ≈ {eps:.3e}")
print(f"theoretical best h ≈ (3ε)^(1/3) ≈ {h_opt_estimate:.3e}")
print("The best h observed in the h-sweep experiment lies between 1e-5 and 1e-7 — the same order of magnitude as the theory predicts.")

## Parameter experiment: the best range for h

Shrink `h` step by step from 1e-1 to 1e-15, printing `abs(numeric_derivative(np.sin, 1.0, h) - np.cos(1.0))` at each step; watch the error fall and then rise in a U-shaped curve, and find the best step size, about 1e-5 to 1e-7.

- **h too large (1e-1 to 1e-3)**: truncation error dominates, the error is about h²/6·f'''(x), and it falls quadratically as h shrinks.
- **h at its best (1e-5 to 1e-7)**: truncation error and floating-point rounding error balance, and the total error is at its minimum, about 1e-11.
- **h too small (below 1e-12)**: `f(x+h) - f(x-h)` subtracts two nearly equal numbers, the significant digits plunge from 16 to fewer than 4, and the error rises as h shrinks.

In [ ]:
# Switch to another function: is the best h still around 1e-5?
test_funcs = {
    "sin(x) @ x=1.0": (np.sin, 1.0, np.cos(1.0)),
    "x^3   @ x=2.0": (lambda x: x**3, 2.0, 3 * 2.0**2),
    "e^x   @ x=1.0": (np.exp, 1.0, float(np.exp(1.0))),
}
hs_test = [1e-3, 1e-5, 1e-7, 1e-9, 1e-11]
try:
    for name, (fn, x0, true_val) in test_funcs.items():
        errs = [abs(numeric_derivative(fn, x0, h) - true_val) for h in hs_test]
        best_h = hs_test[int(np.argmin(errs))]
        print(f"{name}  best h ≈ {best_h:.0e}  smallest error ≈ {min(errs):.2e}")
except (NotImplementedError, TypeError):
    print("⬜ Finish the numeric_derivative exercise above first, then run this cell")

**Does the best h depend on the function?** Scan three different functions, `sin(x)`, `x³` and `eˣ`, as above, and you'll find the best h lands in the same order of magnitude around 1e-5 every time — no coincidence. Look back at `h_opt ≈ (3ε|f(x)/f'''(x)|)^(1/3)`: the deciding factor is the machine precision `ε` (which has nothing to do with the function), and the function only enters through the ratio `|f(x)/f'''(x)|`, whose swings the cube root "flattens" dramatically — even if `|f(x)/f'''(x)|` changes by a factor of 10, `h_opt` changes only by `10^(1/3)≈2.15`. So for the smooth functions common in Aurora (trig functions, exponentials, polynomials), the default `h=1e-5` is basically always good enough; there's no need to tune it for each function.

## Wrap-up

You can now use `numeric_derivative(f, x, h=1e-5)` to compute the slope of a callable at any point, as a single float with error around 1e-10. This function is used directly in Aurora's gradient check: compare the analytic gradient of the loss with respect to the weights against the numerical gradient, entry by entry, and raise an error if they differ by more than 1e-5. The best range for `h` is around 1e-5: too large and truncation error dominates, too small and floating-point rounding error dominates.

Next lesson: **L23** (gradients) repeats this operation along each parameter dimension to get the gradient vector of a multivariable function.

---
⬇️ **Pass check**: once you've read the wrap-up, finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: numerical derivatives by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Question 1**: f(x) = x³; the analytic derivative is f'(x) = ?, and at x=2, f'(2) = ?

**Question 2**: using the central difference formula (h=0.1), compute by hand the numerical derivative of f(x)=x² at x=1:
CD = (f(1.1) - f(0.9)) / 0.2 = ?

**Question 3**: f(x) = sin(x); f'(0) = ? (from the table of common derivatives)

**Question 4**: as `h` shrinks from 1e-5 to 1e-15, does the numerical error grow or shrink? Why?

Once you've worked them out, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: f'(x) = 3x², f'(2) = 12
f1 = lambda x: x**3
df1_analytical = lambda x: 3 * x**2
assert np.isclose(df1_analytical(2), 12.0)
try:
    nd = numeric_derivative(f1, 2.0)
    assert abs(nd - 12.0) < 1e-7, f"the numerical derivative of x³ at x=2 should be ≈12, got {nd}"
    print(f"Q1 ✅  f(x)=x³, f'(x)=3x², f'(2)={df1_analytical(2)}, numerical check={nd:.8f}")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement numeric_derivative() first, then run the answer-check cell")

# Q2: CD by hand — f(x)=x², x=1, h=0.1
f2 = lambda x: x**2
h = 0.1
cd = (f2(1.1) - f2(0.9)) / (2*h)
assert np.isclose(cd, 2.0, atol=1e-10), f"CD should be 2, got {cd}"
print(f"Q2 ✅  CD = (1.21-0.81)/0.2 = {cd:.4f}  (true value f'(1)=2)")

# Q3: sin'(0) = cos(0) = 1
assert np.isclose(np.cos(0), 1.0)
print(f"Q3 ✅  f(x)=sin(x), f'(0)=cos(0)={np.cos(0):.4f}")

# Q4: the error grows when h is too small (catastrophic cancellation; observe it at a non-zero point where f(x±h) are close, x=1)
errors = []
for h_val in [1e-5, 1e-8, 1e-12, 1e-15]:
    nd_sin = (np.sin(1.0 + h_val) - np.sin(1.0 - h_val)) / (2*h_val)
    errors.append(abs(nd_sin - np.cos(1.0)))
# the best h (1e-5) has the smallest error, 1e-15 the largest
assert errors[0] < errors[-1], "did not observe the error growing as h shrinks"
print(f"Q4 ✅  error at h=1e-5={errors[0]:.2e}, error at h=1e-15={errors[-1]:.2e}")
print("     h too small → floating-point cancellation error ↑; the best h≈1e-5 balances truncation and rounding error")
print("\n🎉 Derivatives whiteboard challenge passed! Numerical differentiation = how a computer feels change, internalized.")

In [ ]:
# ✏️ Self-review for this lesson
l22_review = {
    "central_diff_formula":       None,  # remember CD=(f(x+h)-f(x-h))/(2h)? True/False
    "numeric_derivative_done":    None,  # numeric_derivative implemented and passes its asserts? True/False
    "optimal_h_range":            None,  # know the best h≈1e-5 and why? True/False
    "common_derivatives":         None,  # remember the derivatives of xⁿ/eˣ/sin/cos? True/False
    "whiteboard_passed":          None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l22_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l22_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L22 complete! On to L23: gradients')

---

→ **Next lesson** · [L23 · Gradients](L23_gradients.ipynb)

> Next lesson covers **gradients**: the "steepest uphill" direction of a multivariable function, and computing partial derivatives and the gradient vector.